<a href="https://colab.research.google.com/github/evinracher/3010090-ontological-engineering/blob/main/week7/workshop/GraphRAG_RDFLib_EnMemoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RAG Avanzado con Ontologías OWL en Memoria (rdflib + OWL-RL)

## Tutorial Completo: Inferencia y Zero-Shot Retrieval

✨ **Nueva Versión**: Todo funciona **en memoria** sin necesidad de GraphDB Cloud

### 📚 Tecnologías:
- **Ontologías OWL** para conocimiento estructurado
- **rdflib** para grafos RDF en memoria
- **OWL-RL** para razonamiento automático e inferencias
- **LangChain + LangGraph** para orquestación

### 📚 Casos de Uso:
1. **Inferencia**: Deducir relaciones que no están explícitas en los datos
2. **Zero-Shot Retrieval**: Responder preguntas sobre conceptos que no aparecen textualmente en documentos

### 🎯 Dominio del Ejemplo:
Crearemos una ontología médica sobre **enfermedades metabólicas**, con jerarquías de condiciones, síntomas y tratamientos.

---
## 📦 Paso 1: Instalación de Dependencias

In [ ]:
!pip install -q langchain langchain-community langgraph rdflib owlrl reportlab chromadb sentence-transformers pypdf langchain-google-genai langchain_text_splitters fastembed

print("✅ Dependencias instaladas correctamente")

## 🔐 Configuración de Credenciales

Se necesita API Key de Gemini

In [ ]:
import os
from google.colab import userdata

# API Key para el LLM
api_key = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = api_key

print("\n✅ Credenciales configuradas")

---
## 🏗️ Paso 2: Creación de la Ontología OWL

Vamos a crear una ontología completa sobre **Enfermedades Metabólicas** con:
- **Jerarquía de clases**: Enfermedad → EnfermedadMetabólica → Diabetes → DiabetesTipo1/Tipo2
- **Propiedades**: `tieneSintoma`, `tratadoCon`, `causadaPor`
- **Instancias**: Enfermedades específicas, síntomas, medicamentos
- **Reglas de inferencia**: Para deducir nuevas relaciones

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL
from rdflib.namespace import XSD

# Crear el grafo OWL
g = Graph()

# Definir namespaces
MED = Namespace("http://ejemplo.org/medicina#")
g.bind("med", MED)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

# DEFINICIÓN DE CLASES (JERARQUÍA)

# Clase raíz: Enfermedad
g.add((MED.Enfermedad, RDF.type, OWL.Class))
g.add((MED.Enfermedad, RDFS.label, Literal("Enfermedad", lang="es")))

# Subclase: EnfermedadMetabólica
g.add((MED.EnfermedadMetabolica, RDF.type, OWL.Class))
g.add((MED.EnfermedadMetabolica, RDFS.subClassOf, MED.Enfermedad))
g.add((MED.EnfermedadMetabolica, RDFS.label, Literal("Enfermedad Metabólica", lang="es")))

# Subclase: Diabetes (es un tipo de enfermedad metabólica)
g.add((MED.Diabetes, RDF.type, OWL.Class))
g.add((MED.Diabetes, RDFS.subClassOf, MED.EnfermedadMetabolica))
g.add((MED.Diabetes, RDFS.label, Literal("Diabetes", lang="es")))

# Tipos específicos de Diabetes
g.add((MED.DiabetesTipo1, RDF.type, OWL.Class))
g.add((MED.DiabetesTipo1, RDFS.subClassOf, MED.Diabetes))
g.add((MED.DiabetesTipo1, RDFS.label, Literal("Diabetes Tipo 1", lang="es")))

g.add((MED.DiabetesTipo2, RDF.type, OWL.Class))
g.add((MED.DiabetesTipo2, RDFS.subClassOf, MED.Diabetes))
g.add((MED.DiabetesTipo2, RDFS.label, Literal("Diabetes Tipo 2", lang="es")))

# Otra enfermedad metabólica
g.add((MED.Obesidad, RDF.type, OWL.Class))
g.add((MED.Obesidad, RDFS.subClassOf, MED.EnfermedadMetabolica))
g.add((MED.Obesidad, RDFS.label, Literal("Obesidad", lang="es")))

# Clases auxiliares
g.add((MED.Sintoma, RDF.type, OWL.Class))
g.add((MED.Sintoma, RDFS.label, Literal("Síntoma", lang="es")))

g.add((MED.Medicamento, RDF.type, OWL.Class))
g.add((MED.Medicamento, RDFS.label, Literal("Medicamento", lang="es")))

g.add((MED.FactorRiesgo, RDF.type, OWL.Class))
g.add((MED.FactorRiesgo, RDFS.label, Literal("Factor de Riesgo", lang="es")))


# DEFINICIÓN DE PROPIEDADES

# Propiedad: tieneSintoma
g.add((MED.tieneSintoma, RDF.type, OWL.ObjectProperty))
g.add((MED.tieneSintoma, RDFS.domain, MED.Enfermedad))
g.add((MED.tieneSintoma, RDFS.range, MED.Sintoma))
g.add((MED.tieneSintoma, RDFS.label, Literal("tiene síntoma", lang="es")))

# Propiedad: tratadoCon
g.add((MED.tratadoCon, RDF.type, OWL.ObjectProperty))
g.add((MED.tratadoCon, RDFS.domain, MED.Enfermedad))
g.add((MED.tratadoCon, RDFS.range, MED.Medicamento))
g.add((MED.tratadoCon, RDFS.label, Literal("tratado con", lang="es")))

# Propiedad: causadaPor
g.add((MED.causadaPor, RDF.type, OWL.ObjectProperty))
g.add((MED.causadaPor, RDFS.domain, MED.Enfermedad))
g.add((MED.causadaPor, RDFS.range, MED.FactorRiesgo))
g.add((MED.causadaPor, RDFS.label, Literal("causada por", lang="es")))

# Propiedad: aumentaRiesgoDe (inversa)
g.add((MED.aumentaRiesgoDe, RDF.type, OWL.ObjectProperty))
g.add((MED.aumentaRiesgoDe, RDFS.domain, MED.FactorRiesgo))
g.add((MED.aumentaRiesgoDe, RDFS.range, MED.Enfermedad))
g.add((MED.aumentaRiesgoDe, OWL.inverseOf, MED.causadaPor))

# INSTANCIAS DE SÍNTOMAS

sintomas = [
    ("Poliuria", "Aumento de la frecuencia urinaria"),
    ("Polidipsia", "Sed excesiva"),
    ("Polifagia", "Hambre excesiva"),
    ("PerdidaPeso", "Pérdida de peso inexplicable"),
    ("Fatiga", "Cansancio extremo"),
    ("VisionBorrosa", "Visión borrosa"),
    ("CicatrizacionLenta", "Cicatrización lenta de heridas")
]

for nombre, descripcion in sintomas:
    sintoma_uri = MED[nombre]
    g.add((sintoma_uri, RDF.type, MED.Sintoma))
    g.add((sintoma_uri, RDFS.label, Literal(descripcion, lang="es")))


# INSTANCIAS DE MEDICAMENTOS

medicamentos = [
    ("Insulina", "Insulina (hormona)"),
    ("Metformina", "Metformina"),
    ("Glibenclamida", "Glibenclamida"),
    ("Empagliflozina", "Empagliflozina (inhibidor SGLT2)")
]

for nombre, descripcion in medicamentos:
    med_uri = MED[nombre]
    g.add((med_uri, RDF.type, MED.Medicamento))
    g.add((med_uri, RDFS.label, Literal(descripcion, lang="es")))


# INSTANCIAS DE FACTORES DE RIESGO

factores = [
    ("Sedentarismo", "Estilo de vida sedentario"),
    ("DietaAltaEnAzucar", "Dieta alta en azúcar"),
    ("Genetica", "Predisposición genética"),
    ("EdadAvanzada", "Edad avanzada")
]

for nombre, descripcion in factores:
    factor_uri = MED[nombre]
    g.add((factor_uri, RDF.type, MED.FactorRiesgo))
    g.add((factor_uri, RDFS.label, Literal(descripcion, lang="es")))


# INSTANCIA ESPECÍFICA: Paciente con Diabetes Tipo 2

g.add((MED.DiabetesTipo2_Caso1, RDF.type, MED.DiabetesTipo2))
g.add((MED.DiabetesTipo2_Caso1, RDFS.label, Literal("Caso de Diabetes Tipo 2", lang="es")))

# Síntomas asociados
g.add((MED.DiabetesTipo2_Caso1, MED.tieneSintoma, MED.Poliuria))
g.add((MED.DiabetesTipo2_Caso1, MED.tieneSintoma, MED.Polidipsia))
g.add((MED.DiabetesTipo2_Caso1, MED.tieneSintoma, MED.Fatiga))

# Tratamiento
g.add((MED.DiabetesTipo2_Caso1, MED.tratadoCon, MED.Metformina))

# Factores de riesgo
g.add((MED.DiabetesTipo2_Caso1, MED.causadaPor, MED.Sedentarismo))
g.add((MED.DiabetesTipo2_Caso1, MED.causadaPor, MED.DietaAltaenAzucar))


# INSTANCIA: Diabetes Tipo 1

g.add((MED.DiabetesTipo1_Caso1, RDF.type, MED.DiabetesTipo1))
g.add((MED.DiabetesTipo1_Caso1, MED.tieneSintoma, MED.PerdidaPeso))
g.add((MED.DiabetesTipo1_Caso1, MED.tieneSintoma, MED.Poliuria))
g.add((MED.DiabetesTipo1_Caso1, MED.tieneSintoma, MED.Polifagia))
g.add((MED.DiabetesTipo1_Caso1, MED.tratadoCon, MED.Insulina))
g.add((MED.DiabetesTipo1_Caso1, MED.causadaPor, MED.Genetica))


# REGLAS DE INFERENCIA (PROPIEDADES TRANSITIVAS)

# Si X es subclase de Y, y Y tiene un síntoma S, entonces X también tiene S
# Esto lo manejará GraphDB con razonamiento RDFS/OWL

# Guardar la ontología
g.serialize("/content/ontologia_medicina.owl", format="xml")

print("✅ Ontología creada con éxito!")
print(f"📊 Total de triples: {len(g)}")
print("\n📁 Archivo guardado: ontologia_medicina.owl")

### 🔍 Visualizar la estructura de la ontología

In [ ]:
# Mostrar las clases principales
print("\n🏗️ JERARQUÍA DE CLASES:")
print("="*50)
for s, p, o in g.triples((None, RDFS.subClassOf, None)):
    subclass_label = g.value(s, RDFS.label, default=s.split('#')[-1])
    parent_label = g.value(o, RDFS.label, default=o.split('#')[-1])
    print(f"  • {subclass_label} ⊆ {parent_label}")

print("\n🔗 PROPIEDADES DEFINIDAS:")
print("="*50)
for s in g.subjects(RDF.type, OWL.ObjectProperty):
    label = g.value(s, RDFS.label, default=s.split('#')[-1])
    print(f"  • {label}")

---
## 🔥 Paso 3: Cargar Ontología en Memoria y Aplicar Razonamiento OWL-RL

Aquí es donde la magia sucede:
1. Cargamos la ontología en un grafo rdflib en memoria
2. Aplicamos razonamiento OWL-RL para generar **inferencias automáticas**
3. El razonador deduce nuevos triples que no estaban explícitos

**Ejemplo de inferencia:**
- Explícito en la ontología: `DiabetesTipo1 subClassOf Diabetes`
- Explícito en la ontología: `Diabetes subClassOf EnfermedadMetabolica`
- **Inferencia automática**: `DiabetesTipo1 subClassOf EnfermedadMetabolica` ✨

In [ ]:
from rdflib import Graph
import owlrl

# Crear grafo en memoria y cargar la ontología
print("📥 Cargando ontología en memoria...")
ontology_graph = Graph()
ontology_graph.parse("/content/ontologia_medicina.owl", format="xml")

print(f"✅ Ontología cargada: {len(ontology_graph)} triples")

# Aplicar razonamiento OWL-RL
print("\n🧠 Aplicando razonamiento OWL-RL...")
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(ontology_graph)

print(f"✅ Razonamiento completado: {len(ontology_graph)} triples (incluye inferencias)")

# Mostrar cuántas inferencias se generaron
inferencias = len(ontology_graph) - 83  # 83 era el número original
print(f"\n✨ Inferencias generadas: {inferencias} nuevos triples")

---
## 🔧 Función Central: Consultar la Ontología en Memoria

Esta función ejecuta queries SPARQL directamente en memoria usando rdflib.

In [ ]:
def query_ontology_local(sparql_query):
    """
    Ejecuta una consulta SPARQL en la ontología en memoria.

    Args:
        sparql_query (str): Query SPARQL a ejecutar

    Returns:
        rdflib.query.Result: Resultados de la query
    """
    try:
        results = ontology_graph.query(sparql_query)
        return results
    except Exception as e:
        print(f"❌ Error en consulta SPARQL: {e}")
        return None

# Prueba rápida
test_query = """
PREFIX med: <http://ejemplo.org/medicina#>
SELECT (COUNT(*) as ?count)
WHERE {
    ?s ?p ?o .
}
"""

print("🧪 Probando consulta SPARQL en memoria...")
test_results = query_ontology_local(test_query)
if test_results:
    for row in test_results:
        print(f"✅ Total de triples accesibles vía SPARQL: {row[0]}")
else:
    print("❌ Error en la consulta de prueba")

print("\n✅ Función query_ontology_local creada y verificada")

---
## 📄 Paso 4: Carga del Documento PDF

Cargar el documento médico sobre diabetes `guia_diabetes`que **NO** menciona explícitamente algunos conceptos que están en la ontología. Esto permitirá demostrar el **zero-shot retrieval**.

In [ ]:
pdf_filename = "/content/guia_diabetes.pdf"
from langchain_community.document_loaders import PyPDFLoader

# Cargar el PDF
print("📖 Cargando documento PDF...")
loader = PyPDFLoader(pdf_filename)
documents = loader.load()

print(f"   Páginas cargadas: {len(documents)}")

---
## 📚 Paso 5: Crear el Vector Store (Base de Conocimiento Textual)

Vamos a procesar el PDF y crear embeddings para búsqueda semántica tradicional.

In [ ]:
## Para ver los modelos de google
import google.generativeai as genai

# List available models
for m in genai.list_models():
    print(m.name)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Dividir en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
texts = text_splitter.split_documents(documents)
print(f"   Chunks creados: {len(texts)}")

# Crear embeddings
print("\n🧮 Creando embeddings...")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

# Crear vector store
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)

print("✅ Vector store creado exitosamente!")

---
## 🧠 Paso 7: CASO 1 - INFERENCIA

**Concepto clave:** La ontología nos permite encontrar términos relacionados que pueden aparecer en el PDF.

**Ejemplo:**
- Usuario pregunta sobre "diabetes"
- La ontología nos dice que diabetes es una "enfermedad metabólica"
- También nos da términos relacionados: síntomas (poliuria, polidipsia), tratamientos (insulina, metformina)
- **Usamos todos estos términos para buscar en el PDF**, no solo "diabetes"

Este enfoque mejora el retrieval porque encontramos documentos relevantes que no mencionan explícitamente "diabetes" pero hablan de sus síntomas o tratamientos.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import re

def inferencia_ontologica_expandir_terminos(pregunta_usuario, vectorstore):
    """
    INFERENCIA ONTOLÓGICA:
    1. Extrae conceptos clave de la pregunta del usuario
    2. Usa la ontología para expandir estos conceptos
    3. Busca en el PDF usando TODOS los términos
    4. Genera respuesta combinando ontología + PDF
    """
    print("\n" + "="*70)
    print("🧬 DEMOSTRACIÓN: INFERENCIA ONTOLÓGICA")
    print("="*70)
    print(f"\n❓ Pregunta del usuario: {pregunta_usuario}")

    llm = ChatGoogleGenerativeAI(
        model="models/gemini-3.1-flash-lite-preview",
        temperature=0.1
    )

    def extract_text_from_llm(content):
        """Helper to extract string from potential dict/list response"""
        if isinstance(content, list) and len(content) > 0:
            content = content[0]
        if isinstance(content, dict) and 'text' in content:
            return str(content['text'])
        return str(content)

    # Paso 1: Extraer concepto
    extraction_prompt = f"""
    De la siguiente pregunta médica, extrae el CONCEPTO MÉDICO PRINCIPAL (una o dos palabras):
    Pregunta: {pregunta_usuario}
    Responde SOLO con el concepto.
    """

    res_extraction = llm.invoke(extraction_prompt).content
    concepto = extract_text_from_llm(res_extraction).strip().lower()
    print(f"\n🎯 Concepto extraído: '{concepto}'")

    # Paso 2: Generar query SPARQL
    expansion_prompt = f"""
    Genera una query SPARQL para encontrar información relacionada con: \"{concepto}\"

    La query debe obtener:
    1. Clases padre (superclases) del concepto
    2. Clases hija (subclases) del concepto
    3. Síntomas asociados (si aplica)
    4. Tratamientos asociados (si aplica)
    5. Factores de riesgo asociados (si aplica)

    Usa estos prefijos:
    PREFIX med: <http://ejemplo.org/medicina#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    El concepto puede estar en med:Diabetes, med:DiabetesTipo1, med:DiabetesTipo2,
    med:EnfermedadMetabolica, med:Obesidad, etc.

    IMPORTANTE: Busca variaciones del concepto (por ejemplo, si es \"diabetes\",
    busca med:Diabetes, med:DiabetesTipo1, med:DiabetesTipo2)

    Responde SOLO con la query SPARQL.
    """

    print("\n📊 PASO 1: Expansión de términos usando la ontología")
    res_expansion = llm.invoke(expansion_prompt).content
    sparql_expansion = extract_text_from_llm(res_expansion).strip()

    if '```' in sparql_expansion:
        sparql_expansion = sparql_expansion.split('```')[1]
        if sparql_expansion.startswith('sparql\n'):
            sparql_expansion = sparql_expansion[7:]
        sparql_expansion = sparql_expansion.strip()

    print(f"Query SPARQL para expansión:\n{sparql_expansion}\n")

    # Ejecutar query
    resultados_ontologia = query_ontology_local(sparql_expansion)

    # Paso 3: Extraer términos
    terminos_expandidos = {concepto}
    if resultados_ontologia:
        print("✅ Términos relacionados encontrados en la ontología:")
        for row in resultados_ontologia:
            for item in row:
                if item:
                    texto = str(item)
                    if '#' in texto:
                        termino = texto.split('#')[-1]
                        termino_legible = re.sub(r'(?<!^)(?=[A-Z])', ' ', termino).lower()
                        terminos_expandidos.add(termino_legible)
                    elif 3 < len(texto) < 50:
                        terminos_expandidos.add(texto.lower())

        # FILTRAR términos genéricos
        terminos_expandidos = {t for t in terminos_expandidos if t not in ['thing', 'nothing', 'owl:thing', 'owl:nothing']}

        # Mostrar términos expandidos (limitar a los más relevantes)
        terminos_relevantes = [t for t in terminos_expandidos if len(t) > 3][:10]
        for termino in sorted(terminos_relevantes):
            print(f"  • {termino}")

    # Paso 4: Búsqueda Vectorial
    query_vectorial = " ".join(list(terminos_expandidos)[:5])
    docs_encontrados = vectorstore.similarity_search(query_vectorial, k=5)
    contexto_pdf = "\n\n".join([doc.page_content for doc in docs_encontrados])

    contexto_ontologia = "\n".join(terminos_expandidos) if terminos_expandidos else "No se encontró información relevante"

    # Paso 5: Respuesta final
    final_prompt = f"""
    Eres un asistente médico experto.

    PREGUNTA DEL USUARIO: {pregunta_usuario}

    TÉRMINOS RELACIONADOS (de la ontología médica):
    {contexto_ontologia}

    INFORMACIÓN DEL DOCUMENTO:
    {contexto_pdf}

    Proporciona una respuesta completa y precisa.

    IMPORTANTE: Explica cómo usaste la inferencia ontológica:
    - Menciona que partiste del concepto '{concepto}'
    - Menciona qué términos relacionados encontraste en la ontología
    - Explica cómo estos términos te ayudaron a encontrar información relevante en el documento
    """
    respuesta = llm.invoke(final_prompt)

    content_final = extract_text_from_llm(respuesta.content)

    print("\n✅ RESPUESTA FINAL CON INFERENCIA")
    print(content_final)

    return {
        'concepto_original': concepto,
        'terminos_expandidos': list(terminos_expandidos),
        'docs_pdf': [doc.page_content for doc in docs_encontrados],
        'respuesta': content_final
    }

In [ ]:
print("\n" + "="*70)
print("EJEMPLO: Inferencia Ontológica (Expansión de Términos)")
print("="*70)

resultado_inferencia = inferencia_ontologica_expandir_terminos(
    pregunta_usuario="¿Qué sabes sobre las enfermedades metabólicas?",
    vectorstore=vectorstore
)

print("\n📊 Análisis del resultado:")
print(f"  - Concepto original: {resultado_inferencia['concepto_original']}")
print(f"  - Términos expandidos: {len(resultado_inferencia['terminos_expandidos'])}")
print(f"  - Documentos PDF encontrados: {len(resultado_inferencia['docs_pdf'])}")

---
## 🎯 Zero-Shot Retrieval: Respuestas SOLO desde la Ontología

**Concepto clave:** Responder preguntas usando ÚNICAMENTE el conocimiento estructurado de la ontología, SIN consultar el PDF.

**Ejemplo:**
- Usuario pregunta: "¿Qué medicamento se usa para la resistencia a la insulina?"
- Esta información específica NO está en el PDF
- PERO está en la ontología: `DiabetesTipo2_Caso1 tratadoCon Metformina`
- **Respondemos directamente desde la ontología**, sin buscar en el PDF

**Diferencia con Inferencia:**
- **Inferencia**: Usa la ontología para EXPANDIR la búsqueda en el PDF
- **Zero-Shot**: Usa SOLO la ontología, ignora el PDF completamente

In [ ]:
import rdflib
from rdflib import URIRef, Literal as RDFLiteral
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


# 0. Cargar la ontología
grafo_ontologia = rdflib.Graph()
# Asegúrate de tener el archivo en esta ruta, o cámbiala según corresponda.
grafo_ontologia.parse("/content/ontologia_medicina.owl", format="xml")

# --- 1. Definición del Estado del Agente ---
class AgentState(TypedDict):
    # 'add_messages' asegura que los mensajes se concatenen en lugar de sobrescribirse
    messages: Annotated[list, add_messages]

# --- 2. Base de Datos ---
def query_ontology_local(query: str) -> str:
    print(f"\n[🔍 Ejecutando SPARQL en RDFlib...]\n{query}\n")

    try:
        resultados = grafo_ontologia.query(query)

        if len(resultados) == 0:
            return "La consulta se ejecutó correctamente pero no arrojó ningún resultado."

        filas_texto = []

        for fila in resultados:
            elementos_fila = []

            for var in resultados.vars:
                valor = fila[var]

                if valor is not None:
                    if isinstance(valor, URIRef):
                        texto_limpio = str(valor).split('#')[-1]
                        if '/' in texto_limpio:
                            texto_limpio = texto_limpio.split('/')[-1]
                    # ¡Aquí usamos el alias RDFLiteral!
                    elif isinstance(valor, RDFLiteral):
                        texto_limpio = str(valor.value)
                    else:
                        texto_limpio = str(valor)

                    elementos_fila.append(f"{var}: {texto_limpio}")

            if elementos_fila:
                filas_texto.append(", ".join(elementos_fila))

        resultado_final = "Resultados de la base de datos:\n"
        for i, fila in enumerate(filas_texto, 1):
            resultado_final += f"{i}. {fila}\n"

        print(f"[✅ Resultados extraídos:]\n{resultado_final}")
        return resultado_final

    except Exception as e:
        error_msg = f"Error de ejecución SPARQL: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

# --- 3. Definición de Herramientas (Tools) ---
@tool
def consultar_clases_ontologia(sparql_query: str) -> str:
    """
    Útil EXCLUSIVAMENTE para consultar la JERARQUÍA de la ontología.
    Ejemplo: "¿Qué tipos de enfermedades metabólicas existen?", "¿Cuáles son las subclases de Enfermedad?".
    Usa patrones como: ?x rdf:type/rdfs:subClassOf* med:ClaseBase
    """
    try:
        resultados = query_ontology_local(sparql_query)
        if not resultados:
            return "No se encontraron resultados de clases. Revisa los URIs o haz la consulta más general y vuelve a intentarlo."
        return str(resultados)
    except Exception as e:
        return f"Error de sintaxis SPARQL: {str(e)}. Corrige la query y ejecuta la herramienta nuevamente."

@tool
def consultar_propiedades_ontologia(sparql_query: str) -> str:
    """
    Útil EXCLUSIVAMENTE para consultar PROPIEDADES, RELACIONES e INSTANCIAS.
    Ejemplo: "Síntomas de la diabetes", "Tratamientos para la obesidad", "Factores de riesgo".
    Usa propiedades como: med:tieneSintoma, med:tratadoCon, med:causadaPor.
    """
    try:
        resultados = query_ontology_local(sparql_query)
        if not resultados:
            return "No se encontraron relaciones o propiedades. Verifica si la enfermedad o propiedad existe y vuelve a intentarlo."
        return str(resultados)
    except Exception as e:
        return f"Error de sintaxis SPARQL: {str(e)}. Corrige la query y ejecuta la herramienta nuevamente."

tools = [consultar_clases_ontologia, consultar_propiedades_ontologia]

# --- 4. Configuración del Modelo y Prompt ---
# Usamos bind_tools para que Gemini sepa qué funciones puede ejecutar
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.1
).bind_tools(tools)

# Contexto unificado que se le pasa al agente como regla base
CONTEXTO_SISTEMA = """
Eres un asistente médico inteligente respaldado por una ontología OWL.
Tu objetivo es responder a las preguntas del usuario buscando información en la ontología a través de herramientas SPARQL, y luego explicar los resultados en lenguaje natural claro y empático.

ONTOLOGÍA DISPONIBLE:
- Namespace: med = <http://ejemplo.org/medicina#>
- Prefijos obligatorios en tus queries: rdfs, rdf, med.
- CLASES PRINCIPALES: med:Enfermedad, med:EnfermedadMetabolica, med:Diabetes, med:DiabetesTipo1, med:DiabetesTipo2, med:Obesidad, med:Sintoma, med:Medicamento, med:FactorRiesgo.
- PROPIEDADES: med:tieneSintoma, med:tratadoCon, med:causadaPor, med:aumentaRiesgoDe.

REGLAS DE ACTUACIÓN:
1. Analiza la pregunta del usuario.
2. Si es sobre jerarquías o tipos de enfermedades, diseña una query SPARQL y usa 'consultar_clases_ontologia'.
3. Si es sobre síntomas, tratamientos o causas, diseña una query SPARQL y usa 'consultar_propiedades_ontologia'.
4. SIEMPRE incluye en tus queries: OPTIONAL { ?variable rdfs:label ?label } y usa un LIMIT 20.
5. Si una herramienta devuelve un error o vacío, MODIFICA tu query y vuelve a llamar a la herramienta. No te rindas al primer error.
6. CRÍTICO: Tu respuesta final al usuario debe ser en LENGUAJE NATURAL puro. ESTÁ ESTRICTAMENTE PROHIBIDO mostrar la consulta SPARQL, URIs, prefijos (med:) o código OWL al usuario. Habla como un médico explicando a su paciente.
"""

# --- 5. Lógica de los Nodos de LangGraph ---
def nodo_agente(state: AgentState):
    """Llama al LLM con el historial de mensajes y el system prompt."""
    messages = state["messages"]

    # Aseguramos que el prompt del sistema siempre esté al principio
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=CONTEXTO_SISTEMA)] + messages

    response = llm.invoke(messages)
    return {"messages": [response]}

# Comprueba si el modelo decidió usar una herramienta o si ya dio una respuesta final
def condicion_continuar(state: AgentState) -> Literal["tools", "__end__"]:
    mensajes = state["messages"]
    ultimo_mensaje = mensajes[-1]

    # Si el modelo generó llamadas a herramientas, vamos al nodo de herramientas
    if hasattr(ultimo_mensaje, 'tool_calls') and ultimo_mensaje.tool_calls:
        return "tools"
    # De lo contrario, terminó
    return "__end__"

# --- 6. Construcción del Grafo ---
workflow = StateGraph(AgentState)

# Agregar nodos
workflow.add_node("agent", nodo_agente)
workflow.add_node("tools", ToolNode(tools)) # ToolNode maneja automáticamente la ejecución

# Definir el flujo (Edges)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", condicion_continuar)
workflow.add_edge("tools", "agent")

# Compilar el agente
agente_ontologia = workflow.compile()

# --- 7. Función Principal para interactuar ---
def consultar_agente(pregunta_usuario: str):
    print(f"\n👤 Usuario: {pregunta_usuario}")

    inputs = {"messages": [HumanMessage(content=pregunta_usuario)]}

    # Stream permite ver el progreso (útil para debugging)
    for event in agente_ontologia.stream(inputs, stream_mode="values"):
        ultimo_mensaje = event["messages"][-1]

        if hasattr(ultimo_mensaje, 'tool_calls') and ultimo_mensaje.tool_calls:
             print(f"🤖 Agente está usando la herramienta: {ultimo_mensaje.tool_calls[0]['name']}...")

    # La respuesta final
    respuesta_final = event["messages"][-1].content[0]['text']
    print(f"\n🩺 Asistente: {respuesta_final}\n")
    return respuesta_final

# --- PRUEBA ---
if __name__ == "__main__":
    consultar_agente("diabetes y sus sintomas")